# INF6243 – Projet de session : Détection d'images *deepfake*
**Université du Québec en Outaouais – INF 6243 Techniques d'apprentissage**

**Objectif :** distinguer les visages **réels** des visages **générés ou modifiés par IA** (*deepfakes*)
et comparer plusieurs classificateurs : des modèles classiques sur des caractéristiques extraites par un
CNN pré-entraîné, un perceptron multicouche, un CNN entraîné de zéro et un CNN affiné (*fine-tuning*).

**Jeu de données :** [Deepfake and Real Images – Kaggle (manjilkarki)](https://www.kaggle.com/datasets/manjilkarki/deepfake-and-real-images),
environ 190 000 visages en 256×256 répartis en `Train` / `Validation` / `Test`, classes `Fake` / `Real`.

> **Exécution dans Google Colab** : *Exécution → Modifier le type d'exécution → GPU* (T4, L4 ou A100), puis
> *Exécution → Tout exécuter*. Le jeu de données est téléchargé automatiquement à l'étape 1.

**Plan**
0. Configuration
1. Téléchargement et chargement des données
2. Analyse exploratoire (EDA)
3. Prétraitement et extraction de caractéristiques
4. Modèles classiques (GridSearchCV + validation croisée 5 plis)
5. Réseaux de neurones : CNN de zéro et EfficientNet-B0 affiné
6. Comparaison et sélection du modèle (validation)
7. Évaluation finale sur le test
8. Courbes d'apprentissage
9. Importance des caractéristiques et Grad-CAM
10. Analyse des erreurs
11. Conclusion
12. Tester vos propres images

## 0. Configuration
Par défaut, le notebook utilise le **jeu de données complet** (≈ 140 000 images d'entraînement,
39 000 de validation, 11 000 de test). Prévoir environ 1 h 30 à 3 h sur GPU A100 ou L4, davantage sur T4.
**Reprise automatique** : les caractéristiques extraites et les CNN entraînés sont enregistrés dans Google Drive.
Si la session Colab se déconnecte, relancez *Tout exécuter* : les étapes déjà faites sont rechargées au lieu
d'être recalculées. Pour un essai rapide, indiquez par exemple `N_TRAIN_PER_CLASS = 5000`.

In [ ]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
SMOKE = os.environ.get("DEEPFAKE_SMOKE") == "1"     # test rapide hors Colab (données factices)

CFG = dict(
    N_TRAIN_PER_CLASS = None,     # images par classe pour l'entraînement (None = jeu complet)
    N_VAL_PER_CLASS   = None,     # images par classe pour la validation (None = jeu complet)
    N_TEST_PER_CLASS  = None,     # None = tout le test
    N_GRID            = 20_000,   # sous-échantillon pour GridSearchCV (modèle final réentraîné sur tout le train)
    NYSTROEM_DIM      = 2_000,    # dimension de l'approximation du noyau RBF (SVM)
    PCA_DIM           = 256,      # réduction de dimension des embeddings ResNet-50 (2048 -> 256)
    IMG_SIZE          = 224,
    BATCH             = 128,
    EPOCHS            = 5,        # epochs max (arrêt précoce)
    GRID_EPOCHS       = 1,        # epochs par configuration lors de la recherche d'hyperparamètres
    N_GRID_IMAGES     = 10_000,   # images utilisées pour la recherche d'hyperparamètres des CNN
    PRETRAINED        = True,
    SEED              = 42,
    SAVE_TO_DRIVE     = True,     # copie figures / résultats / modèles dans Google Drive
)
if SMOKE:
    CFG.update(N_TRAIN_PER_CLASS=40, N_VAL_PER_CLASS=20, N_TEST_PER_CLASS=20, N_GRID=60, PCA_DIM=8, NYSTROEM_DIM=20,
               IMG_SIZE=64, BATCH=16, EPOCHS=2, GRID_EPOCHS=1, N_GRID_IMAGES=40, PRETRAINED=False,
               SAVE_TO_DRIVE=False)
CFG

In [ ]:
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)

import time, random, warnings, itertools, copy
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision, torchvision.transforms as T
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.kernel_approximation import Nystroem
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold, learning_curve
from sklearn.inspection import permutation_importance
from sklearn.base import clone
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score, confusion_matrix, roc_curve, auc)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
SEED = CFG["SEED"]
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"
NUM_WORKERS = 0 if SMOKE else min(8, os.cpu_count() or 2)
print("Device :", DEVICE, torch.cuda.get_device_name(0) if AMP else "")
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| CPU :", os.cpu_count())

# Dossier de sortie (Google Drive si disponible)
OUT = Path("outputs")
if IN_COLAB and CFG["SAVE_TO_DRIVE"]:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/INF6243_Deepfake")
FIG, RES, MOD = OUT / "figures", OUT / "resultats", OUT / "modeles"
for d in (FIG, RES, MOD): d.mkdir(parents=True, exist_ok=True)
def save(fig, name): fig.savefig(FIG / f"{name}.png", dpi=150, bbox_inches="tight")
def strat_sample(df, keys, n):
    # au plus n lignes par groupe (compatible pandas 2 et 3)
    return pd.concat([g.sample(min(len(g), n), random_state=SEED) for _, g in df.groupby(keys)])
print("Sorties ->", OUT.resolve())

## 1. Téléchargement et chargement des données
Le téléchargement utilise **`kagglehub`**. Le jeu de données est public : aucun identifiant n'est
normalement nécessaire. Si Kaggle demande une authentification, la cellule propose `kagglehub.login()`
(votre nom d'utilisateur Kaggle et une clé API créée dans *Kaggle → Settings → API*). Vous pouvez aussi
définir les secrets Colab `KAGGLE_USERNAME` et `KAGGLE_KEY` (icône 🔑 à gauche).

In [ ]:
DATASET_URL = "https://www.kaggle.com/datasets/manjilkarki/deepfake-and-real-images"
DATASET = "manjilkarki/deepfake-and-real-images"

if os.environ.get("DEEPFAKE_DATA_DIR"):              # données déjà présentes (test local)
    DATA_ROOT = Path(os.environ["DEEPFAKE_DATA_DIR"])
else:
    import kagglehub
    if IN_COLAB:                                     # secrets Colab facultatifs
        try:
            from google.colab import userdata
            os.environ.setdefault("KAGGLE_USERNAME", userdata.get("KAGGLE_USERNAME"))
            os.environ.setdefault("KAGGLE_KEY", userdata.get("KAGGLE_KEY"))
        except Exception:
            pass
    t0 = time.time()
    try:
        DATA_ROOT = Path(kagglehub.dataset_download(DATASET))
    except Exception as e:
        print("Téléchargement anonyme refusé :", e, "\n-> connexion Kaggle")
        kagglehub.login()
        DATA_ROOT = Path(kagglehub.dataset_download(DATASET))
    print(f"Téléchargé en {time.time() - t0:.0f} s")
print("Racine des données :", DATA_ROOT)

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SPLIT_ALIASES = {"train": "train", "training": "train", "validation": "validation", "val": "validation",
                 "valid": "validation", "test": "test", "testing": "test"}

def find_splits(root):
    # Repère les dossiers Train/Validation/Test contenant des sous-dossiers Fake et Real
    found = {}
    for d in sorted(Path(root).rglob("*")):
        if d.is_dir() and d.name.lower() in SPLIT_ALIASES:
            subs = {s.name.lower(): s for s in d.iterdir() if s.is_dir()}
            if "fake" in subs and "real" in subs:
                found.setdefault(SPLIT_ALIASES[d.name.lower()], subs)
    return found

splits = find_splits(DATA_ROOT)
assert splits, f"Aucun dossier Train/Validation/Test avec Fake/Real trouvé sous {DATA_ROOT}"
rows = []
for split, subs in splits.items():
    for cls in ("real", "fake"):
        for p in subs[cls].iterdir():
            if p.suffix.lower() in IMG_EXT:
                rows.append((str(p), split, cls))
full = pd.DataFrame(rows, columns=["path", "split", "label"])
full["y"] = (full["label"] == "fake").astype(int)            # 1 = deepfake, 0 = réel
print("Splits trouvés :", list(splits))
pd.crosstab(full["split"], full["label"], margins=True)

## 2. Analyse exploratoire
### 2.1 Distribution des classes et valeurs manquantes

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(data=full, x="split", hue="label", order=["train", "validation", "test"],
              hue_order=["real", "fake"], palette={"real": "tab:green", "fake": "tab:red"}, ax=ax)
for c in ax.containers: ax.bar_label(c, fontsize=8)
ax.set(title="Distribution des classes par partition (jeu complet)", xlabel="Partition", ylabel="Nombre d'images")
ax.legend(title="Classe"); save(fig, "01_distribution_classes"); plt.show()

# « Valeurs manquantes » pour des images = fichiers illisibles ou corrompus (vérifiés sur un échantillon)
def check(paths):
    bad, sizes, modes = [], [], []
    for p in paths:
        try:
            with Image.open(p) as im:
                im.verify()
            with Image.open(p) as im:
                sizes.append(im.size); modes.append(im.mode)
        except Exception:
            bad.append(p)
    return bad, sizes, modes
samp = strat_sample(full, ["split", "label"], 300)
bad, sizes, modes = check(samp["path"])
print(f"Fichiers corrompus dans l'échantillon : {len(bad)} / {len(samp)}")
print("Tailles d'image :", pd.Series(sizes).value_counts().head(5).to_dict())
print("Modes couleur   :", pd.Series(modes).value_counts().to_dict())

### 2.2 Exemples d'images

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(16, 4.6))
for r, lab in enumerate(["real", "fake"]):
    for c, p in enumerate(full[(full.label == lab) & (full.split == "train")].sample(8, random_state=SEED)["path"]):
        axes[r, c].imshow(Image.open(p).convert("RGB")); axes[r, c].axis("off")
    axes[r, 0].set_title(f"Classe : {lab}", loc="left", fontsize=12)
fig.suptitle("Exemples d'images d'entraînement (haut : réelles, bas : deepfakes)")
plt.tight_layout(); save(fig, "02_exemples"); plt.show()

## 3. Prétraitement
- **Partitions** : nous conservons les partitions officielles `Train` / `Validation` / `Test`, ce qui
  évite qu'un même visage se retrouve dans deux partitions. Nous en tirons un **sous-échantillon équilibré**
  (même nombre d'images par classe) pour contenir le temps de calcul.
- **Nettoyage** : conversion en RGB, fichiers illisibles exclus (vérification en 2.1).
- **Encodage de la cible** : `fake` = 1, `real` = 0.
- **Normalisation** : redimensionnement à 224×224, pixels dans [0, 1], puis standardisation avec la
  moyenne et l'écart-type d'ImageNet (attendue par les réseaux pré-entraînés). Les caractéristiques des
  modèles classiques sont standardisées (`StandardScaler`) dans les pipelines.
- **Augmentation** (entraînement des CNN seulement) : retournement horizontal et léger recadrage. Elle reste
  volontairement faible pour ne pas effacer les artefacts de génération (textures, hautes fréquences).

In [ ]:
def subsample(df, split, n):
    d = df[df.split == split]
    if n is None: return d.reset_index(drop=True)
    return strat_sample(d, "label", n).sample(frac=1, random_state=SEED).reset_index(drop=True)

train_df = subsample(full, "train", CFG["N_TRAIN_PER_CLASS"])
val_df   = subsample(full, "validation", CFG["N_VAL_PER_CLASS"])
test_df  = subsample(full, "test", CFG["N_TEST_PER_CLASS"])
pd.DataFrame({n: {"images": len(d), "% fake": round(d.y.mean() * 100, 1)}
              for n, d in [("train", train_df), ("validation", val_df), ("test", test_df)]}).T

In [ ]:
S = CFG["IMG_SIZE"]
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
eval_tf  = T.Compose([T.Resize((S, S)), T.ToTensor()])
train_tf = T.Compose([T.RandomResizedCrop(S, scale=(0.85, 1.0), ratio=(0.95, 1.05)),
                      T.RandomHorizontalFlip(), T.ToTensor()])

class ImageDS(Dataset):
    def __init__(self, df, tf):
        self.paths, self.y, self.tf = df["path"].tolist(), df["y"].to_numpy(), tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        with Image.open(self.paths[i]) as im:
            return self.tf(im.convert("RGB")), int(self.y[i])

def loader(df, tf=eval_tf, shuffle=False):
    return DataLoader(ImageDS(df, tf), batch_size=CFG["BATCH"], shuffle=shuffle, num_workers=NUM_WORKERS,
                      pin_memory=AMP, persistent_workers=NUM_WORKERS > 0)

### 3.1 Extraction de caractéristiques
Un seul passage sur les images calcule deux représentations :
1. **Embeddings ResNet-50** (pré-entraîné sur ImageNet, poids gelés) : vecteur de 2048 dimensions issu de
   la dernière couche de *pooling*. Les modèles classiques s'appuient sur cette représentation riche.
2. **Caractéristiques manuelles interprétables** (13) : statistiques de couleur, saturation, netteté
   (variance du laplacien), densité de contours et **répartition de l'énergie spectrale** (FFT) en basses,
   moyennes et hautes fréquences. Les générateurs (GAN, diffusion) laissent souvent des traces dans le
   spectre (Durall et al., 2020 ; Frank et al., 2020).

In [ ]:
HC_NAMES = ["moy_R", "moy_G", "moy_B", "std_R", "std_G", "std_B", "saturation", "var_laplacien",
            "densite_contours", "energie_BF", "energie_MF", "energie_HF", "ratio_HF_BF"]
LAP = torch.tensor([[0., 1., 0.], [1., -4., 1.], [0., 1., 0.]]).view(1, 1, 3, 3)
SOBX = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]]).view(1, 1, 3, 3)

def radial_grid(n, device):
    yy, xx = torch.meshgrid(torch.arange(n, device=device), torch.arange(n, device=device), indexing="ij")
    return torch.sqrt((yy - n / 2) ** 2 + (xx - n / 2) ** 2) / (n / 2 * np.sqrt(2))   # rayon dans [0, 1]

@torch.no_grad()
def handcrafted(x):
    # x : (B, 3, S, S) dans [0, 1] -> (B, 13) caractéristiques, (B, S, S) log-spectre
    gray = (0.299 * x[:, 0] + 0.587 * x[:, 1] + 0.114 * x[:, 2]).unsqueeze(1)
    mx, mn = x.max(1).values, x.min(1).values
    sat = ((mx - mn) / (mx + 1e-6)).mean((1, 2))
    lap = F.conv2d(gray, LAP.to(x.device), padding=1).flatten(1).var(1)
    gx = F.conv2d(gray, SOBX.to(x.device), padding=1)
    gy = F.conv2d(gray, SOBX.transpose(2, 3).to(x.device), padding=1)
    edges = ((gx ** 2 + gy ** 2).sqrt() > 0.5).float().mean((1, 2, 3))
    spec = torch.fft.fftshift(torch.fft.fft2(gray.squeeze(1)), dim=(-2, -1))
    power = spec.abs() ** 2
    r = radial_grid(gray.shape[-1], x.device)
    tot = power.sum((1, 2)) + 1e-8
    lo, mid, hi = [(power * m).sum((1, 2)) / tot for m in (r < 0.1, (r >= 0.1) & (r < 0.4), r >= 0.4)]
    feats = torch.stack([*x.mean((2, 3)).T, *x.std((2, 3)).T, sat, lap, edges, lo, mid, hi,
                         torch.log10(hi / (lo + 1e-8) + 1e-12)], 1)
    return feats, torch.log1p(spec.abs())

def build_backbone():
    w = torchvision.models.ResNet50_Weights.IMAGENET1K_V2 if CFG["PRETRAINED"] else None
    m = torchvision.models.resnet50(weights=w); m.fc = nn.Identity()
    return m.eval().to(DEVICE)

@torch.no_grad()
def extract(df, backbone, name):
    embs, hcs, spec_sum, spec_n = [], [], {0: 0, 1: 0}, {0: 0, 1: 0}
    t0 = time.time()
    for x, y in loader(df):
        x = x.to(DEVICE, non_blocking=True)
        with torch.autocast(DEVICE.type, enabled=AMP):
            e = backbone((x - MEAN.to(DEVICE)) / STD.to(DEVICE))
        h, s = handcrafted(x.float())
        embs.append(e.float().cpu()); hcs.append(h.cpu())
        for c in (0, 1):
            m = (y == c)
            if m.any():
                spec_sum[c] = spec_sum[c] + s[m.to(DEVICE)].sum(0).cpu(); spec_n[c] += int(m.sum())
    print(f"{name:10s} {len(df):6d} images en {time.time() - t0:.0f} s")
    spec = {c: (spec_sum[c] / max(spec_n[c], 1)).numpy() for c in (0, 1)}
    return torch.cat(embs).numpy(), torch.cat(hcs).numpy(), spec

y_tr, y_va, y_te = train_df.y.to_numpy(), val_df.y.to_numpy(), test_df.y.to_numpy()
# Cache : la clé dépend de la configuration -> changer les tailles force une nouvelle extraction
key = f"n{CFG['N_TRAIN_PER_CLASS']}_{CFG['N_VAL_PER_CLASS']}_{CFG['N_TEST_PER_CLASS']}_s{S}_p{int(CFG['PRETRAINED'])}"
CACHE = OUT / f"features_{key}.npz"
if CACHE.exists():
    z = np.load(CACHE)
    assert (z["y_tr"] == y_tr).all() and (z["y_te"] == y_te).all(), "cache incohérent : supprimez le fichier"
    E_tr, E_va, E_te = (z[k].astype(np.float32) for k in ("E_tr", "E_va", "E_te"))
    H_tr, H_va, H_te = z["H_tr"], z["H_va"], z["H_te"]
    spec_tr = {0: z["spec0"], 1: z["spec1"]}
    print("Caractéristiques rechargées depuis", CACHE)
else:
    backbone = build_backbone()
    E_tr, H_tr, spec_tr = extract(train_df, backbone, "train")
    E_va, H_va, _ = extract(val_df, backbone, "validation")
    E_te, H_te, _ = extract(test_df, backbone, "test")
    np.savez(CACHE, E_tr=E_tr.astype(np.float16), E_va=E_va.astype(np.float16), E_te=E_te.astype(np.float16),
             H_tr=H_tr, H_va=H_va, H_te=H_te, spec0=spec_tr[0], spec1=spec_tr[1], y_tr=y_tr, y_va=y_va, y_te=y_te)
    del backbone; torch.cuda.empty_cache() if AMP else None
print("Embeddings :", E_tr.shape, "| caractéristiques manuelles :", H_tr.shape)

### 2.3 (suite de l'EDA) Statistiques descriptives et distributions des caractéristiques

In [ ]:
hc = pd.DataFrame(H_tr, columns=HC_NAMES).assign(classe=np.where(y_tr == 1, "fake", "real"))
display(hc.groupby("classe").describe().T.round(4))
fig, axes = plt.subplots(2, 4, figsize=(17, 7))
for ax, col in zip(axes.ravel(), ["moy_R", "std_R", "saturation", "var_laplacien", "densite_contours",
                                  "energie_MF", "energie_HF", "ratio_HF_BF"]):
    sns.histplot(data=hc, x=col, hue="classe", stat="density", common_norm=False, element="step", bins=40,
                 palette={"real": "tab:green", "fake": "tab:red"}, ax=ax)
    ax.set(title=f"Distribution : {col}", xlabel=col, ylabel="Densité")
plt.tight_layout(); save(fig, "03_histogrammes_caracteristiques"); plt.show()

fig, axes = plt.subplots(1, 4, figsize=(17, 4))
for ax, col in zip(axes, ["var_laplacien", "energie_HF", "ratio_HF_BF", "saturation"]):
    sns.boxplot(data=hc, x="classe", y=col, hue="classe", palette={"real": "tab:green", "fake": "tab:red"},
                ax=ax, legend=False, showfliers=False)
    ax.set(title=f"Boxplot : {col}", xlabel="Classe", ylabel=col)
plt.tight_layout(); save(fig, "04_boxplots"); plt.show()

In [ ]:
corr = hc.drop(columns="classe").assign(y=y_tr).corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax, cbar_kws={"label": "Corrélation de Pearson"})
ax.set_title("Corrélations entre caractéristiques manuelles (y = 1 si deepfake)")
save(fig, "05_correlations"); plt.show()
print(corr["y"].drop("y").sort_values(key=abs, ascending=False).round(3))

In [ ]:
# Spectre de Fourier moyen par classe et profil radial (signature fréquentielle)
def radial_profile(img, nbins=60):
    n = img.shape[0]; yy, xx = np.indices(img.shape)
    r = np.sqrt((yy - n / 2) ** 2 + (xx - n / 2) ** 2)
    bins = np.linspace(0, r.max(), nbins + 1); idx = np.digitize(r.ravel(), bins) - 1
    return bins[:-1] / r.max(), np.bincount(idx, img.ravel(), nbins + 1)[:nbins] / np.maximum(np.bincount(idx, minlength=nbins + 1)[:nbins], 1)

fig, axes = plt.subplots(1, 4, figsize=(19, 4.3))
for ax, c, lab in [(axes[0], 0, "réelles"), (axes[1], 1, "deepfakes")]:
    im = ax.imshow(spec_tr[c], cmap="magma"); ax.axis("off")
    ax.set_title(f"Log-spectre FFT moyen : images {lab}"); fig.colorbar(im, ax=ax, label="log(1+|F|)")
diff = spec_tr[1] - spec_tr[0]
im = axes[2].imshow(diff, cmap="RdBu_r", vmin=-np.abs(diff).max(), vmax=np.abs(diff).max()); axes[2].axis("off")
axes[2].set_title("Différence (fake − réel)"); fig.colorbar(im, ax=axes[2], label="Δ log-amplitude")
for c, lab, col in [(0, "réel", "tab:green"), (1, "fake", "tab:red")]:
    f, p = radial_profile(spec_tr[c]); axes[3].plot(f, p, label=lab, color=col)
axes[3].set(title="Profil radial du spectre", xlabel="Fréquence spatiale normalisée", ylabel="log-amplitude moyenne")
axes[3].legend(title="Classe")
plt.tight_layout(); save(fig, "06_spectre_fft"); plt.show()

In [ ]:
# Projection 2D des embeddings ResNet-50 (PCA) : les classes sont-elles séparables ?
Z = PCA(2, random_state=SEED).fit_transform(StandardScaler().fit_transform(E_tr))
idx = np.random.RandomState(SEED).choice(len(Z), min(4000, len(Z)), replace=False)
fig, ax = plt.subplots(figsize=(6.5, 5))
for c, lab, col in [(0, "réel", "tab:green"), (1, "fake", "tab:red")]:
    m = idx[y_tr[idx] == c]; ax.scatter(Z[m, 0], Z[m, 1], s=4, alpha=0.4, c=col, label=lab)
ax.set(title="Embeddings ResNet-50 projetés par PCA (2 composantes)", xlabel="CP 1", ylabel="CP 2")
ax.legend(title="Classe", markerscale=3); save(fig, "07_pca_embeddings"); plt.show()

## 4. Modèles classiques

| Modèle | Fonctionnement | Justification |
|---|---|---|
| **Régression logistique** | Modèle linéaire, P(fake) = σ(w·x + b), régularisation L2 (`C`). | Référence simple et rapide ; mesure à quel point les embeddings sont **linéairement** séparables (*linear probe*). |
| **SVM à noyau RBF (Nyström)** | Maximise la marge dans l'espace induit par un noyau gaussien (`C`, `gamma`). Le noyau est approché par la méthode de Nyström (2000 composantes), puis un SVM linéaire est appliqué. | Frontières non linéaires. Un SVM RBF exact coûte O(n²) à O(n³) et serait impraticable sur 140 000 images ; l'approximation de Nyström (Williams & Seeger, 2001) le rend utilisable sur le jeu complet. |
| **Random Forest** | Ensemble d'arbres sur échantillons bootstrap, vote majoritaire. | Robuste, peu de prétraitement, fournit l'**importance des caractéristiques** (utilisée sur les caractéristiques manuelles). |
| **KNN** | Classe majoritaire parmi les *k* plus proches voisins (distance euclidienne après standardisation et PCA). | Méthode non paramétrique : teste si des images similaires dans l'espace CNN partagent la même classe. |
| **Perceptron multicouche (MLP)** | Réseau dense (1 à 2 couches cachées, ReLU) entraîné par rétropropagation (Adam). | Premier modèle neuronal, sur les embeddings : non linéaire et peu coûteux. |

**Pipeline** : `StandardScaler → PCA(256) → classificateur` sur les embeddings ResNet-50. Le PCA est
ajusté dans chaque pli, ce qui évite toute fuite d'information. **Réglage** : `GridSearchCV`, validation croisée
**stratifiée à 5 plis**, critère **F1**, sur un sous-échantillon de `N_GRID` images. Le meilleur modèle est
réentraîné sur tout l'ensemble d'entraînement.
Deux modèles supplémentaires (Régression logistique et Random Forest) sont entraînés sur les **13 caractéristiques
manuelles** pour interpréter les indices exploités.

In [ ]:
P = CFG["PCA_DIM"]
def emb_pipe(clf): return Pipeline([("sc", StandardScaler()), ("pca", PCA(P, random_state=SEED)), ("clf", clf)])
def hc_pipe(clf):  return Pipeline([("sc", StandardScaler()), ("clf", clf)])

CLASSICAL = {
    "Régression logistique": ("emb", emb_pipe(LogisticRegression(max_iter=3000)),
                              {"clf__C": [0.01, 0.1, 1, 10]}),
    "SVM RBF (Nyström)":     ("emb", Pipeline([("sc", StandardScaler()), ("pca", PCA(P, random_state=SEED)),
                                               ("rbf", Nystroem(kernel="rbf", n_components=CFG["NYSTROEM_DIM"], random_state=SEED)),
                                               ("clf", LinearSVC(max_iter=5000, random_state=SEED))]),
                              {"rbf__gamma": [1 / (4 * P), 1 / P], "clf__C": [0.1, 1]}),
    "Random Forest":         ("emb", emb_pipe(RandomForestClassifier(n_jobs=-1, min_samples_leaf=2, random_state=SEED)),
                              {"clf__n_estimators": [300], "clf__max_depth": [None, 30],
                               "clf__max_features": ["sqrt", 0.1]}),
    "KNN":                   ("emb", emb_pipe(KNeighborsClassifier()),
                              {"clf__n_neighbors": [5, 15, 31], "clf__weights": ["uniform", "distance"]}),
    "MLP (embeddings)":      ("emb", emb_pipe(MLPClassifier(max_iter=300, early_stopping=True, random_state=SEED)),
                              {"clf__hidden_layer_sizes": [(256,), (512, 128)], "clf__alpha": [1e-4, 1e-2]}),
    "Rég. log. (manuelles)": ("hc", hc_pipe(LogisticRegression(max_iter=3000)), {"clf__C": [0.1, 1, 10]}),
    "RF (manuelles)":        ("hc", hc_pipe(RandomForestClassifier(n_jobs=-1, random_state=SEED)),
                              {"clf__n_estimators": [300], "clf__max_depth": [None, 12], "clf__min_samples_leaf": [1, 5]}),
}
FEATS = {"emb": (E_tr, E_va, E_te), "hc": (H_tr, H_va, H_te)}
CV = StratifiedKFold(5, shuffle=True, random_state=SEED)
SCORING = {"accuracy": "accuracy", "precision": "precision", "recall": "recall", "f1": "f1"}
grid_idx = np.random.RandomState(SEED).permutation(len(y_tr))[:CFG["N_GRID"]]

def scores_of(m, X):
    return m.predict_proba(X)[:, 1] if hasattr(m, "predict_proba") else m.decision_function(X)

def metrics(y, pred, score=None):
    d = {"Accuracy": accuracy_score(y, pred), "Précision (fake)": precision_score(y, pred, zero_division=0),
         "Rappel (fake)": recall_score(y, pred), "F1 (fake)": f1_score(y, pred),
         "F1 macro": f1_score(y, pred, average="macro")}
    if score is not None: d["ROC-AUC"] = roc_auc_score(y, score)
    return d

best, cv_rows, cv_folds, val_rows, grid_info, val_scores = {}, {}, {}, {}, {}, {}
for name, (fs, pipe, grid) in CLASSICAL.items():
    Xtr, Xva, _ = FEATS[fs]
    t0 = time.time()
    gs = GridSearchCV(pipe, grid, cv=CV, scoring=SCORING, refit="f1", n_jobs=-1)
    gs.fit(Xtr[grid_idx], y_tr[grid_idx])
    r, i = gs.cv_results_, gs.best_index_
    model = clone(gs.best_estimator_).fit(Xtr, y_tr)            # réentraînement sur tout le train
    best[name] = model
    cv_rows[name] = {f"CV {k}": r[f"mean_test_{k}"][i] for k in SCORING} | {"CV f1 (écart-type)": r["std_test_f1"][i]}
    cv_folds[name] = [r[f"split{k}_test_f1"][i] for k in range(5)]
    val_scores[name] = scores_of(model, Xva)
    val_rows[name] = metrics(y_va, model.predict(Xva), val_scores[name])
    grid_info[name] = {"caractéristiques": fs, "combinaisons": len(r["params"]),
                       "temps (s)": round(time.time() - t0, 1), "meilleurs hyperparamètres": gs.best_params_}
    print(f"{name:24s} F1-CV={r['mean_test_f1'][i]:.4f}  F1-val={val_rows[name]['F1 (fake)']:.4f}  ({time.time() - t0:.0f}s)")

In [ ]:
display(pd.DataFrame(grid_info).T)
pd.DataFrame(cv_rows).T.round(4)

## 5. Réseaux de neurones convolutifs

| Modèle | Fonctionnement | Justification |
|---|---|---|
| **CNN de zéro** | 4 blocs (convolution 3×3 → BatchNorm → ReLU → max-pooling), *pooling* global puis couche dense. Entraîné à partir de poids aléatoires. | Montre ce qu'un réseau apprend sans connaissances préalables. Les premières couches peuvent capter directement les artefacts locaux de génération. |
| **EfficientNet-B0 affiné** | Réseau pré-entraîné sur ImageNet (convolutions MBConv + *squeeze-and-excitation*). Toutes les couches sont réentraînées (*fine-tuning*) avec une nouvelle tête binaire. | Approche de l'état de l'art en détection de deepfakes (Rössler et al., 2019 ; Tan & Le, 2019) : le transfert d'apprentissage donne de bonnes performances avec peu d'epochs. |

**Entraînement** : perte d'entropie croisée binaire, optimiseur AdamW, taux d'apprentissage en cosinus,
précision mixte (AMP) sur GPU, **arrêt précoce** sur la perte de validation (patience = 2).
**Réglage des hyperparamètres** : recherche sur grille du taux d'apprentissage (et de la régularisation
*weight decay*) sur `N_GRID_IMAGES` images, `GRID_EPOCHS` epoch(s) par configuration, évaluée en F1 sur la validation.
La validation croisée k-fold n'est pas appliquée aux CNN : son coût serait de 5 entraînements complets.
L'ensemble de validation, distinct et de grande taille, sert d'estimation.

In [ ]:
class Normalize(nn.Module):
    def __init__(self, net): super().__init__(); self.net = net
    def forward(self, x): return self.net((x - MEAN.to(x.device)) / STD.to(x.device)).squeeze(-1)

def small_cnn():
    def block(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                                          nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                                          nn.MaxPool2d(2))
    return Normalize(nn.Sequential(block(3, 32), block(32, 64), block(64, 128), block(128, 256),
                                   nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(256, 1)))

def efficientnet():
    w = torchvision.models.EfficientNet_B0_Weights.IMAGENET1K_V1 if CFG["PRETRAINED"] else None
    m = torchvision.models.efficientnet_b0(weights=w)
    m.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.classifier[1].in_features, 1))
    return Normalize(m)

@torch.no_grad()
def predict(model, dl):
    model.eval(); ps, ys = [], []
    for x, y in dl:
        with torch.autocast(DEVICE.type, enabled=AMP):
            ps.append(torch.sigmoid(model(x.to(DEVICE, non_blocking=True)).float()).cpu()); ys.append(y)
    return torch.cat(ps).numpy(), torch.cat(ys).numpy()

def train(model, tr_df, va_df, lr, wd=1e-4, epochs=CFG["EPOCHS"], patience=2, verbose=True):
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    tr_dl, va_dl = loader(tr_df, train_tf, shuffle=True), loader(va_df)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(tr_dl), pct_start=0.15)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=AMP)
    loss_fn = nn.BCEWithLogitsLoss()
    hist = {k: [] for k in ("train_loss", "val_loss", "train_acc", "val_acc")}
    best_loss, best_state, bad = np.inf, None, 0
    for ep in range(epochs):
        t0 = time.time(); model.train(); tl, tc, n = 0.0, 0, 0
        for x, y in tr_dl:
            x, y = x.to(DEVICE, non_blocking=True), y.float().to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE.type, enabled=AMP):
                out = model(x); loss = loss_fn(out.float(), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tl += loss.item() * len(y); tc += ((out > 0).float() == y).sum().item(); n += len(y)
        p, yv = predict(model, va_dl)
        vl = F.binary_cross_entropy(torch.tensor(p).clamp(1e-6, 1 - 1e-6), torch.tensor(yv).float()).item()
        hist["train_loss"].append(tl / n); hist["train_acc"].append(tc / n)
        hist["val_loss"].append(vl); hist["val_acc"].append(((p >= 0.5) == yv).mean())
        if verbose:
            print(f"  epoch {ep + 1}/{epochs}  train_loss={tl / n:.4f} acc={tc / n:.4f} | "
                  f"val_loss={vl:.4f} acc={hist['val_acc'][-1]:.4f}  ({time.time() - t0:.0f}s)")
        if vl < best_loss - 1e-4:
            best_loss, bad, best_state = vl, 0, copy.deepcopy(model.state_dict())
        else:
            bad += 1
            if bad >= patience: print("  arrêt précoce"); break
    model.load_state_dict(best_state); hist["best_epoch"] = int(np.argmin(hist["val_loss"])) + 1
    return model, hist

def lr_search(builder, grid, name):
    sub = train_df.sample(min(CFG["N_GRID_IMAGES"], len(train_df)), random_state=SEED)
    rows = []
    for lr, wd in itertools.product(grid["lr"], grid["wd"]):
        torch.manual_seed(SEED); t0 = time.time()
        m, _ = train(builder(), sub, val_df, lr, wd, epochs=CFG["GRID_EPOCHS"], patience=99, verbose=False)
        p, yv = predict(m, loader(val_df))
        rows.append({"modèle": name, "lr": lr, "weight_decay": wd, "F1 val": f1_score(yv, p >= 0.5),
                     "temps (s)": round(time.time() - t0)})
        print(f"{name}: lr={lr:g} wd={wd:g} -> F1 val={rows[-1]['F1 val']:.4f}")
        del m; torch.cuda.empty_cache() if AMP else None
    return pd.DataFrame(rows).sort_values("F1 val", ascending=False)

In [ ]:
CNNS = {"CNN de zéro":          (small_cnn,    {"lr": [1e-3, 3e-3], "wd": [1e-4, 1e-2]}),
        "EfficientNet-B0 affiné": (efficientnet, {"lr": [1e-4, 3e-4, 1e-3], "wd": [1e-4, 1e-2]})}
cnn_models, cnn_hist, cnn_grid = {}, {}, []
for name, (builder, grid) in CNNS.items():
    ckpt = MOD / f"{name.replace(' ', '_')}_{key}.pt"
    if ckpt.exists():                                   # reprise après déconnexion
        c = torch.load(ckpt, map_location=DEVICE, weights_only=False)
        m = builder().to(DEVICE); m.load_state_dict(c["state"]); h, g = c["hist"], c["grid"]
        print(f"{name} : modèle rechargé depuis {ckpt}")
    else:
        g = lr_search(builder, grid, name)
        lr, wd = g.iloc[0]["lr"], g.iloc[0]["weight_decay"]
        print(f"\n=== {name} : entraînement final (lr={lr:g}, wd={wd:g}) ===")
        torch.manual_seed(SEED)
        m, h = train(builder(), train_df, val_df, lr, wd)
        torch.save({"state": m.state_dict(), "hist": h, "grid": g}, ckpt)
    cnn_grid.append(g); cnn_models[name], cnn_hist[name] = m, h
    val_scores[name], _ = predict(m, loader(val_df))
    val_rows[name] = metrics(y_va, (val_scores[name] >= 0.5).astype(int), val_scores[name])
cnn_grid = pd.concat(cnn_grid); cnn_grid.round(4)

In [ ]:
fig, axes = plt.subplots(len(cnn_hist), 2, figsize=(12, 4 * len(cnn_hist)), squeeze=False)
for row, (name, h) in zip(axes, cnn_hist.items()):
    ep = np.arange(1, len(h["train_loss"]) + 1)
    row[0].plot(ep, h["train_loss"], "o-", label="Entraînement"); row[0].plot(ep, h["val_loss"], "o-", label="Validation")
    row[0].axvline(h["best_epoch"], ls="--", c="grey", label="meilleur epoch")
    row[0].set(title=f"{name} : perte", xlabel="Epoch", ylabel="Entropie croisée binaire")
    row[1].plot(ep, h["train_acc"], "o-", label="Entraînement"); row[1].plot(ep, h["val_acc"], "o-", label="Validation")
    row[1].set(title=f"{name} : accuracy", xlabel="Epoch", ylabel="Accuracy")
    for a in row: a.legend()
plt.tight_layout(); save(fig, "08_courbes_cnn"); plt.show()

## 6. Comparaison et sélection du modèle (validation)
Le test n'est **pas** utilisé pour choisir le modèle.

In [ ]:
val_df_res = pd.DataFrame(val_rows).T.rename_axis("Modèle").sort_values("F1 (fake)", ascending=False)
display(val_df_res.round(4))
display(pd.DataFrame(cv_rows).T.rename_axis("Modèle").round(4))

def bar_compare(res, title, fname):
    long = res[["Accuracy", "Précision (fake)", "Rappel (fake)", "F1 (fake)"]].reset_index().melt(
        id_vars="Modèle", var_name="Métrique", value_name="Score")
    fig, ax = plt.subplots(figsize=(14, 5))
    sns.barplot(data=long, x="Modèle", y="Score", hue="Métrique", ax=ax)
    ax.set_ylim(max(0, long.Score.min() - 0.05), 1.0)
    ax.set(title=title, xlabel="Modèle", ylabel="Score")
    ax.legend(title="Métrique", bbox_to_anchor=(1.01, 1), loc="upper left")
    for c in ax.containers: ax.bar_label(c, fmt="%.3f", fontsize=7, padding=1)
    plt.xticks(rotation=20, ha="right"); plt.tight_layout(); save(fig, fname); plt.show()

bar_compare(val_df_res, "Comparaison des modèles : ensemble de validation", "09_comparaison_validation")

fig, ax = plt.subplots(figsize=(11, 4))
cvf = pd.DataFrame(cv_folds)
sns.boxplot(data=cvf, ax=ax, color="lightsteelblue"); sns.stripplot(data=cvf, ax=ax, color="black", size=4)
ax.set(title="Validation croisée 5 plis : F1 (fake) par modèle classique", xlabel="Modèle", ylabel="F1 (fake)")
plt.xticks(rotation=20, ha="right"); plt.tight_layout(); save(fig, "10_cv_boxplot"); plt.show()

BEST = val_df_res["F1 (fake)"].idxmax()
print("Modèle retenu (meilleur F1 sur la validation) :", BEST)

## 7. Évaluation finale sur l'ensemble de test

In [ ]:
test_scores, test_preds, test_rows = {}, {}, {}
for name, m in best.items():
    X = FEATS[CLASSICAL[name][0]][2]
    test_scores[name] = scores_of(m, X); test_preds[name] = m.predict(X)
te_dl = loader(test_df)
for name, m in cnn_models.items():
    p, _ = predict(m, te_dl); test_scores[name] = p; test_preds[name] = (p >= 0.5).astype(int)
for name in test_preds:
    test_rows[name] = metrics(y_te, test_preds[name], test_scores[name])
test_res = pd.DataFrame(test_rows).T.rename_axis("Modèle").sort_values("F1 (fake)", ascending=False)
display(test_res.round(4))
bar_compare(test_res, "Comparaison des modèles : ensemble de test", "11_comparaison_test")

In [ ]:
names = list(test_preds); nc = 3; nr = int(np.ceil(len(names) / nc))
fig, axes = plt.subplots(nr, nc, figsize=(13, 3.8 * nr)); axes = axes.ravel()
for ax, n in zip(axes, names):
    cm = confusion_matrix(y_te, test_preds[n])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["real", "fake"], yticklabels=["real", "fake"])
    ax.set(title=n, xlabel="Classe prédite", ylabel="Classe réelle")
for ax in axes[len(names):]: ax.axis("off")
fig.suptitle("Matrices de confusion (test)", fontsize=14); plt.tight_layout(); save(fig, "12_matrices_confusion"); plt.show()

fig, ax = plt.subplots(figsize=(7, 6))
for n in names:
    fpr, tpr, _ = roc_curve(y_te, test_scores[n]); ax.plot(fpr, tpr, label=f"{n} (AUC={auc(fpr, tpr):.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=1, label="hasard")
ax.set(title="Courbes ROC (test)", xlabel="Taux de faux positifs", ylabel="Taux de vrais positifs")
ax.legend(fontsize=8, loc="lower right"); save(fig, "13_roc"); plt.show()

## 8. Courbes d'apprentissage (modèles classiques)
Score F1 en entraînement et en validation croisée selon le nombre d'images.

In [ ]:
lc_models = ["Régression logistique", "SVM RBF (Nyström)", "Random Forest", "MLP (embeddings)"]
fig, axes = plt.subplots(1, 4, figsize=(20, 4.2))
for ax, n in zip(axes, lc_models):
    fs = CLASSICAL[n][0]
    sizes, tr, va = learning_curve(clone(best[n]), FEATS[fs][0][grid_idx], y_tr[grid_idx], cv=CV, scoring="f1",
                                   train_sizes=np.linspace(0.15, 1.0, 6), shuffle=True,
                                   random_state=SEED, n_jobs=-1)
    for s, lab, col in [(tr, "Entraînement", "tab:blue"), (va, "Validation croisée", "tab:orange")]:
        ax.plot(sizes, s.mean(1), "o-", c=col, label=lab)
        ax.fill_between(sizes, s.mean(1) - s.std(1), s.mean(1) + s.std(1), color=col, alpha=0.15)
    ax.set(title=f"Courbe d'apprentissage : {n}", xlabel="Nombre d'images d'entraînement", ylabel="F1 (fake)")
    ax.legend(loc="lower right")
plt.tight_layout(); save(fig, "14_courbes_apprentissage"); plt.show()

## 9. Importance des caractéristiques et Grad-CAM
- **Caractéristiques manuelles** : importance par réduction d'impureté (Random Forest) et **importance par
  permutation** sur la validation (baisse du F1 lorsqu'une caractéristique est mélangée).
- **Grad-CAM** (Selvaraju et al., 2017) sur EfficientNet-B0 : zones du visage qui pèsent le plus dans la décision.

In [ ]:
rf = best["RF (manuelles)"]
imp = pd.Series(rf.named_steps["clf"].feature_importances_, index=HC_NAMES).sort_values()
perm = permutation_importance(rf, H_va, y_va, scoring="f1", n_repeats=10, random_state=SEED, n_jobs=-1)
pimp = pd.Series(perm.importances_mean, index=HC_NAMES).reindex(imp.index)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(imp.index, imp.values, color="tab:purple", label="réduction d'impureté")
axes[0].set(title="Random Forest (manuelles) : importance des caractéristiques", xlabel="Importance", ylabel="Caractéristique")
axes[1].barh(pimp.index, pimp.values, xerr=pd.Series(perm.importances_std, index=HC_NAMES).reindex(imp.index),
             color="tab:orange", label="baisse moyenne du F1")
axes[1].set(title="Importance par permutation (validation)", xlabel="Baisse du F1", ylabel="Caractéristique")
for a in axes: a.legend(loc="lower right")
plt.tight_layout(); save(fig, "15_importance_caracteristiques"); plt.show()

In [ ]:
class GradCAM:
    def __init__(self, model, layer):
        self.model, self.acts, self.grads = model, None, None
        layer.register_forward_hook(lambda m, i, o: setattr(self, "acts", o))
        layer.register_full_backward_hook(lambda m, gi, go: setattr(self, "grads", go[0]))
    def __call__(self, x):
        self.model.eval(); self.model.zero_grad()
        out = self.model(x); out.sum().backward()
        w = self.grads.mean((2, 3), keepdim=True)
        cam = F.relu((w * self.acts).sum(1, keepdim=True))
        cam = F.interpolate(cam, x.shape[-2:], mode="bilinear", align_corners=False).squeeze(1)
        cam = cam / (cam.flatten(1).max(1).values.view(-1, 1, 1) + 1e-8)
        return torch.sigmoid(out).detach().cpu().numpy(), cam.detach().cpu().numpy()

def show_cam(df_rows, title, fname, k=6):
    effnet = cnn_models["EfficientNet-B0 affiné"].float()
    cam = GradCAM(effnet, effnet.net.features[-1])
    df_rows = df_rows.head(k)
    if len(df_rows) == 0: print("aucune image"); return
    x = torch.stack([eval_tf(Image.open(p).convert("RGB")) for p in df_rows["path"]]).to(DEVICE).requires_grad_(True)
    probs, cams = cam(x)
    fig, axes = plt.subplots(2, len(df_rows), figsize=(2.6 * len(df_rows), 5.6), squeeze=False)
    for j, (img, c, pr, lab) in enumerate(zip(x.detach().cpu(), cams, probs, df_rows["label"])):
        axes[0, j].imshow(img.permute(1, 2, 0)); axes[0, j].set_title(f"réel : {lab}\nP(fake)={pr:.2f}", fontsize=9)
        axes[1, j].imshow(img.permute(1, 2, 0)); axes[1, j].imshow(c, cmap="jet", alpha=0.45)
        axes[0, j].axis("off"); axes[1, j].axis("off")
    axes[1, 0].set_title("Grad-CAM", loc="left", fontsize=9)
    fig.suptitle(title); plt.tight_layout(); save(fig, fname); plt.show()

show_cam(strat_sample(test_df, "label", 3),
         "Grad-CAM EfficientNet-B0 : exemples de test", "16_gradcam_exemples")

## 10. Analyse des erreurs du modèle retenu
- **Faux négatif** : un deepfake classé réel (il passe inaperçu). C'est l'erreur la plus grave pour un outil
  de lutte contre la désinformation.
- **Faux positif** : une vraie image accusée d'être fausse.

In [ ]:
print("Modèle analysé :", BEST)
err = test_df.assign(pred=test_preds[BEST], score=test_scores[BEST])
fp = err[(err.y == 0) & (err.pred == 1)].sort_values("score", ascending=False)
fn = err[(err.y == 1) & (err.pred == 0)].sort_values("score")
print(f"Faux positifs : {len(fp)} | Faux négatifs : {len(fn)} | sur {len(err)} images de test")

def grid_imgs(d, title, fname, k=8):
    d = d.head(k)
    if len(d) == 0: print("aucune image pour :", title); return
    fig, axes = plt.subplots(1, len(d), figsize=(2.3 * len(d), 2.8), squeeze=False)
    for ax, (_, r) in zip(axes[0], d.iterrows()):
        ax.imshow(Image.open(r.path).convert("RGB")); ax.axis("off"); ax.set_title(f"score={r.score:.2f}", fontsize=9)
    fig.suptitle(title); plt.tight_layout(); save(fig, fname); plt.show()
grid_imgs(fp, f"{BEST} : faux positifs les plus confiants (réel → fake)", "17_faux_positifs")
grid_imgs(fn, f"{BEST} : faux négatifs les plus confiants (fake → réel)", "18_faux_negatifs")
if "EfficientNet-B0 affiné" in cnn_models:
    show_cam(pd.concat([fp.head(3), fn.head(3)]), "Grad-CAM sur des erreurs", "19_gradcam_erreurs")

In [ ]:
# Les erreurs ont-elles un profil particulier ? (caractéristiques manuelles des images de test)
hte = pd.DataFrame(H_te, columns=HC_NAMES)
grp = np.select([(err.y == 1) & (err.pred == 1), (err.y == 1) & (err.pred == 0),
                 (err.y == 0) & (err.pred == 0), (err.y == 0) & (err.pred == 1)],
                ["fake détecté (VP)", "fake manqué (FN)", "réel correct (VN)", "réel accusé (FP)"], default="?")
prof = hte.assign(groupe=grp).groupby("groupe")[["var_laplacien", "energie_HF", "ratio_HF_BF", "saturation", "moy_R"]].median()
display(prof.round(4))
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(x=grp, y=hte["ratio_HF_BF"], ax=ax, showfliers=False,
            order=["fake détecté (VP)", "fake manqué (FN)", "réel correct (VN)", "réel accusé (FP)"])
ax.set(title=f"{BEST} : ratio HF/BF selon le type de prédiction", xlabel="Groupe", ylabel="log10(énergie HF / BF)")
plt.tight_layout(); save(fig, "20_erreurs_profil"); plt.show()

In [ ]:
pd.DataFrame(grid_info).T.to_csv(RES / "hyperparametres_classiques.csv")
cnn_grid.to_csv(RES / "hyperparametres_cnn.csv", index=False)
pd.DataFrame(cv_rows).T.round(4).to_csv(RES / "cv.csv")
val_df_res.round(4).to_csv(RES / "validation.csv"); test_res.round(4).to_csv(RES / "test.csv")
print("Figures   :", FIG.resolve()); print("Résultats :", RES.resolve()); print("Modèles   :", MOD.resolve())

## 11. Conclusion
_À rédiger après l'exécution complète, à partir des tableaux `validation.csv` et `test.csv` et des figures._

Pistes de discussion :
- Écart entre les modèles classiques sur embeddings gelés et le CNN affiné : valeur du *fine-tuning*.
- Ce que révèlent le spectre FFT et l'importance des caractéristiques manuelles (artefacts haute fréquence ?).
- Grad-CAM : le modèle regarde-t-il le visage (yeux, bouche, contours) ou l'arrière-plan ?
- Limites : généralisation à d'autres générateurs (GAN vs diffusion), compression JPEG des réseaux sociaux,
  vidéos (cohérence temporelle non exploitée ici).
- Améliorations : jeu d'entraînement complet, modèles plus grands (EfficientNet-B4, ViT), augmentation par
  compression, détection vidéo image par image avec agrégation temporelle.

## 12. Tester vos propres images
Cette section est **autonome** : elle fonctionne même dans une nouvelle session Colab, sans réexécuter le
reste du notebook. Elle recharge l'EfficientNet-B0 affiné enregistré dans Drive à l'étape 5.

**Deux façons de fournir des images :**
1. **Téléversement** : une fenêtre s'ouvre, choisissez une ou plusieurs images (jpg, png).
2. **Dossier Drive** : déposez vos images dans `MyDrive/INF6243_Deepfake/mes_images/`.

**Conseils pour un test juste :** le modèle a appris sur des **visages cadrés de face, centrés, en 256×256**.
Recadrez vos images sur le visage. Une photo de paysage ou un corps entier sera « hors distribution » et le
résultat ne voudra rien dire. Testez des deux côtés : quelques **vraies photos** (les vôtres, avec permission)
et quelques **visages générés** (par exemple ceux du site *thispersondoesnotexist.com*).

In [ ]:
import os, sys, glob
from pathlib import Path
import numpy as np, matplotlib.pyplot as plt
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision, torchvision.transforms as T

IN_COLAB = "google.colab" in sys.modules
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if IN_COLAB:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists(): drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/INF6243_Deepfake")
else:
    BASE = Path(os.environ.get("DEEPFAKE_OUT", "outputs"))

# 1) Recharger le modèle entraîné (le plus récent)
ckpts = sorted(glob.glob(str(BASE / "modeles" / "EfficientNet-B0_affin*.pt")), key=os.path.getmtime)
assert ckpts, f"Aucun modèle trouvé dans {BASE / 'modeles'} : exécutez d'abord les étapes 1 à 5."
IM_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IM_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

class Normalize(nn.Module):
    def __init__(self, net): super().__init__(); self.net = net
    def forward(self, x): return self.net((x - IM_MEAN.to(x.device)) / IM_STD.to(x.device)).squeeze(-1)

net = torchvision.models.efficientnet_b0(weights=None)
net.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(net.classifier[1].in_features, 1))
detector = Normalize(net)
detector.load_state_dict(torch.load(ckpts[-1], map_location=DEV, weights_only=False)["state"])
detector = detector.to(DEV).eval()
print("Modèle chargé :", Path(ckpts[-1]).name)

# 2) Prétraitement identique à l'entraînement : carré centré puis 224×224, pixels dans [0, 1]
def prep(img):
    img = img.convert("RGB"); w, h = img.size; c = min(w, h)
    img = img.crop(((w - c) // 2, (h - c) // 2, (w - c) // 2 + c, (h - c) // 2 + c))
    return T.Compose([T.Resize((224, 224)), T.ToTensor()])(img)

In [ ]:
# 3) Récupérer les images : dossier Drive + téléversement
MY_DIR = BASE / "mes_images"; MY_DIR.mkdir(parents=True, exist_ok=True)
if IN_COLAB:
    from google.colab import files
    print("Choisissez des images (ou annulez pour utiliser seulement le dossier Drive)…")
    try:
        for name, data in files.upload().items():
            (MY_DIR / name).write_bytes(data)
    except Exception as e:
        print("Téléversement ignoré :", e)
paths = sorted(p for p in MY_DIR.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".bmp"})
print(f"{len(paths)} image(s) à tester dans {MY_DIR}")

In [ ]:
# 4) Prédiction + Grad-CAM (zones qui ont le plus influencé la décision)
SEUIL = 0.5          # P(fake) >= SEUIL -> « FAUSSE »

acts, grads = {}, {}
layer = detector.net.features[-1]
layer.register_forward_hook(lambda m, i, o: acts.__setitem__("a", o))
layer.register_full_backward_hook(lambda m, gi, go: grads.__setitem__("g", go[0]))

def analyse(path):
    x = prep(Image.open(path)).unsqueeze(0).to(DEV).requires_grad_(True)
    detector.zero_grad(); out = detector(x); out.sum().backward()
    w = grads["g"].mean((2, 3), keepdim=True)
    cam = F.relu((w * acts["a"]).sum(1, keepdim=True))
    cam = F.interpolate(cam, (224, 224), mode="bilinear", align_corners=False)[0, 0]
    cam = (cam / (cam.max() + 1e-8)).detach().cpu().numpy()
    return x[0].detach().cpu().permute(1, 2, 0).numpy(), float(torch.sigmoid(out).item()), cam

rows = []
for i in range(0, len(paths), 4):                    # 4 images par figure
    chunk = paths[i:i + 4]
    fig, axes = plt.subplots(2, len(chunk), figsize=(3.4 * len(chunk), 7), squeeze=False)
    for j, p in enumerate(chunk):
        img, prob, cam = analyse(p)
        verdict = "FAUSSE (deepfake)" if prob >= SEUIL else "VRAIE"
        rows.append({"image": p.name, "P(fake)": round(prob, 4), "verdict": verdict})
        axes[0, j].imshow(img); axes[0, j].axis("off")
        axes[0, j].set_title(f"{p.name[:22]}\n{verdict}\nP(fake) = {prob:.1%}", fontsize=9,
                             color="tab:red" if prob >= SEUIL else "tab:green")
        axes[1, j].imshow(img); axes[1, j].imshow(cam, cmap="jet", alpha=0.45); axes[1, j].axis("off")
        axes[1, j].set_title("Grad-CAM : zones décisives", fontsize=9)
    plt.tight_layout(); plt.show()

import pandas as pd
resultats_perso = pd.DataFrame(rows)
resultats_perso

**Comment lire le résultat**
- **P(fake)** est la probabilité, estimée par le modèle, que l'image soit générée par IA. Au-dessus de 50 %,
  le verdict est « FAUSSE ». Une valeur proche de 50 % signifie que le modèle **hésite**.
- **Grad-CAM** : les zones rouges sont celles qui ont le plus pesé dans la décision. Si le rouge tombe sur
  l'arrière-plan plutôt que sur le visage, le modèle s'appuie peut-être sur un indice trompeur.
- **Limite importante** : ce jeu de données provient de générateurs précis. Une image produite par un
  générateur plus récent (Midjourney, Stable Diffusion récents, etc.) peut tromper le modèle : c'est le
  problème bien connu de la **généralisation** des détecteurs de deepfakes, à discuter dans le rapport.